# Tensor and autograd drills

Four tasks on the two halves of the lesson. Parts 1 and 2 ban Python loops: a
k-nearest-neighbor classifier and a padded-batch summary, written with tensor
operations alone and checked against a slow loop that is obviously right. Parts
3 and 4 use autograd as a proof assistant: the backward pass of a small network
by hand, confirmed by `.backward()`, and gradient accumulation that reproduces
the full-batch gradient exactly.

**How to work through it.** Each part asks for one or two functions. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first. It holds the data of every part and `uses_loop`, which the checks of
Parts 1 and 2 run on your functions.

In [ ]:
import dis
import torch
import torch.nn as nn
import torch.nn.functional as F


def uses_loop(fn):
    """True if fn, a comprehension inside it, or a function it calls contains a Python for or while loop."""
    names, seen, codes = set(), set(), [fn.__code__]
    while codes:
        code = codes.pop()
        if id(code) in seen:
            continue
        seen.add(id(code))
        for ins in dis.get_instructions(code):
            if ins.opname in ("FOR_ITER", "JUMP_ABSOLUTE", "SETUP_LOOP") or "BACKWARD" in ins.opname:
                return True
        names.update(code.co_names)
        codes += [const for const in code.co_consts if hasattr(const, "co_code")]
        codes += [obj.__code__ for nm in list(names)
                  for obj in [fn.__globals__.get(nm)]
                  if hasattr(obj, "__code__")]
    return False


torch.manual_seed(0)

# Part 1: three clusters of points in the plane, labeled 0, 1 and 2.
centers = torch.tensor([[0.0, 0.0], [3.0, 0.0], [0.0, 3.0]])
y_known = torch.randint(0, 3, (300,))
X_known = centers[y_known] + torch.randn(300, 2)    # the labeled points the classifier knows
y_query = torch.randint(0, 3, (100,))
X_query = centers[y_query] + torch.randn(100, 2)    # the points to classify

# Part 2: six sequences of different lengths, padded to length 10.
lengths = torch.tensor([10, 3, 7, 1, 5, 8])
H_pad = torch.randn(6, 10, 4)
scores_pad = torch.randn(6, 10)
pad = torch.arange(10)[None, :] >= lengths[:, None]
H_pad[pad] = 1000.0          # garbage in the padding, so reading it shows in the answer
scores_pad[pad] = 50.0

# Parts 3 and 4: a two-layer network's data and weights.
X_net = torch.randn(64, 5)
y_net = torch.randint(0, 3, (64,))
W1 = 0.5 * torch.randn(5, 16)
b1 = torch.zeros(16)
W2 = 0.5 * torch.randn(16, 3)
b2 = torch.zeros(3)
print("setup complete")

## Part 1 (coding task): k-nearest neighbors with no loops

A k-nearest-neighbor classifier labels a query point by a vote among the $k$ known points closest
to it. With loops it is two nested loops and a sort. With tensor operations it is a few lines,
and it runs far faster. No Python `for` or `while` loop and no comprehension may appear in this
part: the check reads the bytecode of your functions and rejects them. That rule covers every
helper you write inside them or call from them, because the point of the part is the
vectorized idiom, not a loop hidden one call away.

- `pairwise_sq_dists(A, B)`, for `A` of shape `(n, d)` and `B` of shape `(m, d)`, returns the
  `(n, m)` matrix of squared distances $\lVert\mathbf{a}_i - \mathbf{b}_j\rVert^2$. Do not build
  an `(n, m, d)` tensor, which runs out of memory for large inputs. Expand
  $\lVert\mathbf{a} - \mathbf{b}\rVert^2 = \lVert\mathbf{a}\rVert^2 + \lVert\mathbf{b}\rVert^2 - 2\,\mathbf{a}^\top\mathbf{b}$,
  so that the one large operation is a matrix product, and clamp the result at 0, because
  rounding can push an entry slightly below it.
- `knn_predict(X_train, y_train, X_query, k, n_classes)` returns one predicted label per query
  row. Take the indices of the `k` nearest training points with `topk`, look up their labels,
  tally the labels per class with `scatter_add_` (or with `F.one_hot` and a sum), and return the
  class with the most votes. On a tie, return the smallest label, which is what `argmax` does on
  its own.

The check prints the accuracy on `X_query` for $k = 1, 5, 15$.

In [ ]:
def pairwise_sq_dists(A, B):
    # TODO
    raise NotImplementedError


def knn_predict(X_train, y_train, X_query, k, n_classes):
    # TODO: distances, then the k nearest, then their labels, then votes per class, then argmax
    raise NotImplementedError

In [ ]:
def loop_knn(X_train, y_train, X_query, k, n_classes):
    """The slow reference: two loops and a sort."""
    out = []
    for q in X_query:
        d = [float(((q - x) ** 2).sum()) for x in X_train]
        nearest = sorted(range(len(X_train)), key=lambda i: d[i])[:k]
        counts = [0] * n_classes
        for i in nearest:
            counts[int(y_train[i])] += 1
        out.append(max(range(n_classes), key=lambda c: (counts[c], -c)))
    return torch.tensor(out)


assert not uses_loop(pairwise_sq_dists) and not uses_loop(knn_predict), "no Python loops or comprehensions in this part"
D = pairwise_sq_dists(X_query, X_known)
assert D.shape == (100, 300), f"expected shape (100, 300), got {tuple(D.shape)}"
assert torch.allclose(D, ((X_query[:, None] - X_known[None]) ** 2).sum(-1), atol=1e-4), "the distances do not match the direct computation"
assert (pairwise_sq_dists(X_known, X_known) >= 0).all(), "clamp the rounding errors at 0"

# A size only a memory-light answer survives: the (n, m, d) route would build
# 2000 x 3000 x 64 entries, about 1.5 GB of float32.
A_big = torch.randn(2000, 64)
B_big = torch.randn(3000, 64)
D_big = pairwise_sq_dists(A_big, B_big)
assert D_big.shape == (2000, 3000) and (D_big >= 0).all(), \
    "the (n, m, d) route would build a 2000 x 3000 x 64 tensor here"

for k in (1, 4, 5, 15):
    pred = knn_predict(X_known, y_known, X_query, k, 3)
    assert torch.equal(pred, loop_knn(X_known, y_known, X_query, k, 3)), f"k = {k}: the predictions differ from the loop version"
    if k != 4:
        print(f"k = {k:2d}: accuracy {(pred == y_query).float().mean():.2f}")
print("Part 1 passed")

## Part 2 (coding task): Padded sequences, masks, pooling and the last real token

Sequences in a batch have different lengths, so they are padded to one length $T$, and the
padding must be ignored. `H_pad` has shape `(B, T, d)`, and row `b` is real only in its first
`lengths[b]` positions. The setup filled the padding with garbage, 1000 in `H_pad` and 50 in
`scores_pad`, so an answer that touches it is visibly wrong. Again no loops and no
comprehensions, in this function, in any helper you define inside it, and in any helper you call.

Write `summarize_padded(H, scores, lengths)`, which returns a dictionary of four tensors:

- `"mask"`: the `(B, T)` boolean mask, `True` at the real positions. Build it by comparing
  `torch.arange(T)` with `lengths` through broadcasting.
- `"mean"`: the `(B, d)` mean of each sequence's real vectors.
- `"pooled"`: the `(B, d)` attention pooling $\sum_t \alpha_{bt}\,\mathbf{h}_{bt}$, where
  $\alpha_{b\cdot}$ is the softmax of row `b` of `scores` over its real positions only. Set the
  padded scores to $-\infty$ with `masked_fill` before the softmax, so that their weights come
  out exactly 0.
- `"last"`: the `(B, d)` vector at each sequence's last real position, `lengths[b] - 1`, picked
  out with advanced indexing.

In [ ]:
def summarize_padded(H, scores, lengths):
    batch, T, d = H.shape
    # TODO: mask, mean, pooled, last
    raise NotImplementedError

In [ ]:
def loop_summary(H, scores, lengths):
    means, pooled, last = [], [], []
    for b, n in enumerate(lengths.tolist()):
        real = H[b, :n]
        w = torch.softmax(scores[b, :n], dim=0)
        means.append(real.mean(dim=0))
        pooled.append((w[:, None] * real).sum(dim=0))
        last.append(real[-1])
    return torch.stack(means), torch.stack(pooled), torch.stack(last)


assert not uses_loop(summarize_padded), "no Python loops or comprehensions in this part"
out = summarize_padded(H_pad, scores_pad, lengths)
assert out["mask"].dtype == torch.bool and out["mask"].shape == (6, 10), "mask: a (B, T) boolean tensor"
assert out["mask"].sum(dim=1).tolist() == lengths.tolist(), "mask: row b has lengths[b] True entries"
want_mean, want_pooled, want_last = loop_summary(H_pad, scores_pad, lengths)
for name, want in (("mean", want_mean), ("pooled", want_pooled), ("last", want_last)):
    got = out[name]
    assert got.shape == want.shape, f"{name}: shape {tuple(got.shape)}, expected {tuple(want.shape)}"
    assert not torch.isnan(got).any(), f"{name}: contains nan (0 times inf from the padding?)"
    assert torch.allclose(got, want, atol=1e-5), f"{name}: does not match the loop version (did padding leak in?)"
print(out["last"][:3])
print("Part 2 passed")

## Part 3 (coding task): Backpropagation by hand, checked by autograd

Autograd can check a derivation: derive a gradient by hand, code it, and let `.backward()`
confirm it. The function here is the loss of a two-layer network on `X_net`, of shape
$n \times 5$, with labels `y_net` in $\{0, 1, 2\}$:

$$
Z = XW_1 + \mathbf{b}_1,\quad H = \mathrm{ReLU}(Z),\quad S = HW_2 + \mathbf{b}_2,\quad
\ell = -\frac1n\sum_{i=1}^n \log\mathrm{softmax}(S_i)_{y_i}.
$$

Write `manual_backward(X, y, W1, b1, W2, b2)`, which computes the loss and the gradients of
$\ell$ with respect to $W_1$, $\mathbf{b}_1$, $W_2$ and $\mathbf{b}_2$ with no autograd at all:
no `requires_grad`, no `backward`, no `torch.autograd`. Return `(loss, grads)`, with `grads` a
dictionary keyed `"W1"`, `"b1"`, `"W2"`, `"b2"`.

The first step is given. With $P$ the row-wise softmax of $S$ and $Y$ the one-hot labels, the
gradient of $\ell$ with respect to the scores is $(P - Y)/n$. From there, walk back through the
forward pass line by line with the chain rule. Every gradient has the shape of the tensor
it belongs to, which checks each step as you go.

In [ ]:
def manual_backward(X, y, W1, b1, W2, b2):
    n = X.shape[0]
    # Forward pass, keeping what the backward pass needs.
    # TODO: Z, H, S, P and the loss
    # Backward pass.
    # TODO: dS, then dW2, db2, dH, dZ, dW1, db1
    raise NotImplementedError

In [ ]:
loss, grads = manual_backward(X_net, y_net, W1, b1, W2, b2)

params = {name: t.clone().requires_grad_() for name, t in {"W1": W1, "b1": b1, "W2": W2, "b2": b2}.items()}
ref_loss = F.cross_entropy(F.relu(X_net @ params["W1"] + params["b1"]) @ params["W2"] + params["b2"], y_net)
ref_loss.backward()

assert torch.allclose(loss, ref_loss.detach(), atol=1e-6), f"loss {float(loss):.6f}, autograd says {float(ref_loss):.6f}"
assert set(grads) == {"W1", "b1", "W2", "b2"}, "return all four gradients"
# Cells run through exec, so inspect.getsource cannot read them: the ban reads the
# bytecode's name table instead, the same walk uses_loop uses in the setup cell.
names, seen, codes = set(), set(), [manual_backward.__code__]
while codes:
    code = codes.pop()
    if id(code) in seen:
        continue
    seen.add(id(code))
    names.update(code.co_names)
    codes += [obj.__code__ for nm in list(names)
              for obj in [manual_backward.__globals__.get(nm)]
              if hasattr(obj, "__code__")]
    codes += [c for c in code.co_consts if hasattr(c, "co_code")]
assert not any(tok in name for name in names for tok in ("requires_grad", "backward", "autograd", "enable_grad")), \
    "manual_backward must be derived by hand: no requires_grad, no backward, no torch.autograd"
for name, p in params.items():
    g = grads[name]
    assert not g.requires_grad, f"{name}: computed with autograd"
    assert g.shape == p.shape, f"{name}: gradient shape {tuple(g.shape)}, parameter shape {tuple(p.shape)}"
    gap = (g - p.grad).abs().max().item()
    assert gap < 1e-6, f"{name}: largest difference from autograd is {gap:.2e}"
    print(f"{name}: matches autograd (largest difference {gap:.1e})")
print("Part 3 passed")

## Part 4 (coding task): Gradient accumulation over uneven micro-batches

`.backward()` adds into `.grad` rather than overwriting it. Training code leans on this when a
batch is too big for memory: it runs several smaller micro-batches through `backward()` before
one optimizer step. The sum equals the full-batch gradient only when each micro-batch loss
carries the right weight.

Write `accumulated_grads(model, X, y, chunk_sizes)`. Zero the model's gradients once, cut
`X, y` into consecutive chunks of the given sizes, and call `backward()` once per chunk on that
chunk's mean cross-entropy loss times a weight of your choosing. Return a list holding a copy of
every parameter's `.grad`. The result must equal the gradient of the mean loss over all of `X`
taken as one batch, for any chunk sizes, unequal ones included.

In [ ]:
def accumulated_grads(model, X, y, chunk_sizes):
    model.zero_grad()
    # TODO: one backward per chunk, each loss weighted so that the sum is the full-batch gradient
    raise NotImplementedError

In [ ]:
torch.manual_seed(1)
model = nn.Sequential(nn.Linear(5, 16), nn.ReLU(), nn.Linear(16, 3))
model.zero_grad()
F.cross_entropy(model(X_net), y_net).backward()
full = [p.grad.clone() for p in model.parameters()]

for sizes in ([16, 16, 16, 16], [10, 30, 24], [1, 63]):
    acc = accumulated_grads(model, X_net, y_net, sizes)
    gap = max((a - b).abs().max().item() for a, b in zip(acc, full))
    assert gap < 1e-6, f"chunks {sizes}: differs from the full batch by {gap:.2e}"
    print(f"chunks {sizes}: matches the full-batch gradient")

accumulated_grads(model, X_net, y_net, [32, 32])        # leaves the full-batch gradient in .grad
F.cross_entropy(model(X_net), y_net).backward()         # one more backward, with no zero_grad
assert torch.allclose(model[0].weight.grad, 2 * full[0], atol=1e-6), "without zero_grad the gradients add up"
print("Part 4 passed")

## Solution

### Part 1: k-nearest neighbors with no loops

In [ ]:
def pairwise_sq_dists(A, B):
    sq_a = (A * A).sum(dim=1, keepdim=True)     # squared row norms of A, shape (n, 1)
    sq_b = (B * B).sum(dim=1)                   # squared row norms of B, shape (m,)
    cross = A @ B.T                             # a_i . b_j for every pair, shape (n, m)
    return (sq_a + sq_b - 2 * cross).clamp_min(0)


def knn_predict(X_train, y_train, X_query, k, n_classes):
    dists = pairwise_sq_dists(X_query, X_train)                  # (q, n)
    nearest = dists.topk(k, dim=1, largest=False).indices        # (q, k)
    labels = y_train[nearest]                                    # (q, k)
    votes = torch.zeros(X_query.shape[0], n_classes)
    votes.scatter_add_(1, labels, torch.ones(X_query.shape[0], k))   # one vote per neighbor
    return votes.argmax(dim=1)                                   # the first maximum is the smallest label

**Why it works.** `sq_a` has shape `(n, 1)` and `sq_b` has shape `(m,)`, which broadcasts as
`(1, m)`, so `sq_a + sq_b - 2 * cross` comes out `(n, m)` with no `(n, m, d)` tensor in between:
$O(nm)$ memory instead of $O(nmd)$, and the one heavy operation is a matrix product that runs on
optimized library code. `topk(..., largest=False)` finds the $k$ smallest entries of each row
without sorting the whole row. `y_train[nearest]` is advanced indexing, so an index tensor of
shape `(q, k)` gives a `(q, k)` tensor of labels, and `scatter_add_` adds a 1 at column `label`
of each row, a histogram per query. The check includes $k = 4$, where a query can see a two-two
tie, to confirm that the first maximum wins.

### Part 2: Padded sequences, masks, pooling and the last real token

In [ ]:
def summarize_padded(H, scores, lengths):
    batch, T, d = H.shape
    real = torch.arange(T)[None, :] < lengths[:, None]                # (batch, T), True at real positions
    H_real = H.masked_fill(~real[:, :, None], 0.0)                     # padding zeroed, so it adds nothing
    mean_vec = H_real.sum(dim=1) / lengths[:, None].float()
    alpha = scores.masked_fill(~real, float("-inf")).softmax(dim=1)    # weight exactly 0 on padding
    pooled = (alpha[:, :, None] * H_real).sum(dim=1)
    last_vec = H[torch.arange(batch), lengths - 1]
    return {"mask": real, "mean": mean_vec, "pooled": pooled, "last": last_vec}

**Why it works.** `torch.arange(T)[None, :]` is `(1, T)` and `lengths[:, None]` is `(B, 1)`,
so the comparison broadcasts to `(B, T)` with no loop over rows. After `masked_fill` with
$-\infty$, the softmax gives $e^{-\infty} = 0$ to every padded slot, and the real slots
renormalize among themselves. The padded vectors are zeroed as well before the weighted sum: a
weight of 0 times a finite garbage value is 0, but 0 times `inf` or `nan` is `nan`, and real
padding can hold either. `H[torch.arange(B), lengths - 1]` pairs row `b` with position
`lengths[b] - 1`, one vector per row. Transformers use this mask to keep attention off the
padding, and recurrent models take the last real state as the summary of a sequence.

### Part 3: Backpropagation by hand, checked by autograd

In [ ]:
def manual_backward(X, y, W1, b1, W2, b2):
    n = X.shape[0]
    # Forward pass, keeping what the backward pass needs.
    Z = X @ W1 + b1                       # (n, 16)
    H = Z.clamp_min(0)                    # ReLU
    S = H @ W2 + b2                       # (n, 3)
    P = S.softmax(dim=1)
    loss = -P[torch.arange(n), y].log().mean()
    # Backward pass, one forward line at a time.
    Y = F.one_hot(y, S.shape[1]).float()
    dS = (P - Y) / n                      # the given first step, shape (n, 3)
    dW2 = H.T @ dS                        # (16, 3): adds up the per-example outer products
    db2 = dS.sum(dim=0)                   # (3,): the bias reached every row, so add over rows
    dH = dS @ W2.T                        # (n, 16)
    dZ = dH * (Z > 0).float()             # ReLU lets the gradient through only where Z > 0
    dW1 = X.T @ dZ                        # (5, 16)
    db1 = dZ.sum(dim=0)                   # (16,)
    return loss, {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}

**Why it works.** Each line of the backward pass undoes one line of the forward pass. For
$S = HW_2 + \mathbf{b}_2$, entry $S_{ik} = \sum_j H_{ij}(W_2)_{jk} + (b_2)_k$, so the gradient
with respect to $(W_2)_{jk}$ is $\sum_i H_{ij}\,(\mathrm{d}S)_{ik}$ with $\mathrm{d}S$ the given
first step, which is the matrix product $H^\top \mathrm{d}S$, and the bias, broadcast to every
row, collects the column sums of $\mathrm{d}S$. The same rule in the other direction gives
$\mathrm{d}H = \mathrm{d}S\,W_2^\top$. ReLU has slope 1 where $Z > 0$ and 0 elsewhere, so its
backward step multiplies by that mask. The given first step comes from
$\partial(-\log\mathrm{softmax}(\mathbf{s})_y)/\partial s_k = p_k - [k = y]$, averaged over the
$n$ rows. Autograd performs these same steps, which is why the two agree to rounding error.

### Part 4: Gradient accumulation over uneven micro-batches

In [ ]:
def accumulated_grads(model, X, y, chunk_sizes):
    model.zero_grad()
    start = 0
    for size in chunk_sizes:
        xc, yc = X[start:start + size], y[start:start + size]
        start += size
        chunk_mean = F.cross_entropy(model(xc), yc)       # the mean loss over this chunk alone
        (chunk_mean * size / len(X)).backward()           # its share of the rows is its weight
    return [p.grad.clone() for p in model.parameters()]

**Why it works.** The mean over all $n$ rows splits by chunk:
$\frac{1}{n}\sum_{i=1}^{n}\ell_i = \sum_c \frac{m_c}{n}\,\bar\ell_c$, where $\bar\ell_c$ is the
mean loss of chunk $c$ and $m_c$ its size. Gradients of a sum add, so a backward pass on each
$\frac{m_c}{n}\bar\ell_c$ leaves the full-batch gradient in `.grad`. Scaling every chunk by one
over the number of chunks is the special case of equal sizes: with chunks of 1 and 63 rows it
would give the single row 32 times its fair say, and 63 times the per-row say of a row in the
large chunk. The last lines of the check show the other side of accumulation: a further
`backward()` with no zeroing doubles the gradient, which is why every training step starts with
`optimizer.zero_grad()`.